In [83]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
import json
from IPython.display import Markdown
from datasets import load_dataset
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage,HumanMessage





In [84]:
from huggingface_hub import login

hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    print("HuggingFace API key is not working")
else:
    print("HF KEY looks good so far")    
login(hf_token, add_to_git_credential=True)

HF KEY looks good so far


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [85]:
load_dotenv(override=True)
openai_api_key=os.getenv("OPENAI_API_KEY")
if not openai_api_key:
   print("OpenAI API Key not set properly")
else:
    print(f"OpenAI API Key exists")


openai=OpenAI()
openai_model="gpt-5.4"


OpenAI API Key exists


In [86]:
tools=[
    {
        "type": "function",
        "function": {
            "name": "about_stock",
            "description": "Get the full information about stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "stock_name": {
                        "type": "string",
                        "description": "The name of the stock",
                    },
                    "date":{
                        "type":"string",
                        "description":"Date of the trade for spesific stock"
                    }
                },
                "required": ["stock_name","date"],
                "additionalProperties": False
            }
        }
    },
    {
        "type":"function",
           "function":{
                "name":"about_news",
                "description":"News about any stocks",
                "parameters": {
                "type": "object",
                "properties": {
                    "stock_name": {
                        "type": "string",
                        "description": "The name of the stock",
                    },
                    "date":{
                        "type":"string",
                        "description":"Date of the news for spesific stock"
                    }
                },
                "required": ["stock_name","date"],
                "additionalProperties": False
              }
           }
    }
]

In [87]:
from datetime import datetime
def check_date(date,format="%Y-%m-%d"):
    try:
        date_day=datetime.strptime(date,format)
        weekday=date_day.weekday()
        if weekday<5:
            return True 
        else:
             return False
    except ValueError:
        return False

In [88]:
def check_date_format(date,format="%Y-%m-%d"):
    try:
        datetime.strptime(date,format)
        return True 
    except ValueError:
        return False

In [89]:
print(check_date("2026-03-15"))

False


In [90]:
from massive import RESTClient

massive_api_key=os.getenv("MASSIVE_API")
client = RESTClient(massive_api_key)

def about_stock(name,date):
    if check_date(date)==True:
        request = client.get_daily_open_close_agg(
    name,
    date,
    adjusted="true",)
    else:
       return "Date is not avialable for stock market(either the date is in the weekend or in invalid format)!"
    return request.close,request.open,request.high,request.low,request.volume
   

about_stock("TSLA","2026-03-15")



'Date is not avialable for stock market(either the date is in the weekend or in invalid format)!'

In [91]:

from massive.rest.models import (
    TickerNews,
)

def about_news(name,date):
   if check_date_format(date)==True:
     news = []
     for n in client.list_ticker_news(
	  ticker=name.upper(),
	  published_utc=date,
	  order="asc",
	  limit="20",
	  sort="published_utc",
	  ):
      news.append(n)

   document=[]
   for index, item in enumerate(news):
      n=[{"Time":{item.published_utc},"Publisher":{item.publisher.name},"Title":{item.title},"URL":{item.article_url}}]  
      for sent in item.insights:
        sentiments=[{"Stock":{sent.ticker},"Sentiment":{sent.sentiment},"Reason":{sent.sentiment_reasoning}}]
        n.append(sentiments)
      document.append(n)
   return document
   


In [92]:
about_news("TSLA","2026-03-14")

[[{'Time': {'2026-03-14T07:05:00Z'},
   'Publisher': {'The Motley Fool'},
   'Title': {'Finally, a Little Good News for Tesla Investors -- or Is It?'},
   'URL': {'https://www.fool.com/investing/2026/03/14/finally-a-little-good-news-for-tesla-investors-or/?source=iedfolrf0000001'}},
  [{'Stock': {'TSLA'},
    'Sentiment': {'neutral'},
    'Reason': {"While Tesla showed a 10% month-over-month registration increase in Europe, this was against a very weak prior-year comparison. Year-to-date registrations are essentially flat, and the company continues to face headwinds including aging product lineup, Chinese competition, and cooling EV markets. The rebound is described as 'far from strong or secure.'"}}],
  [{'Stock': {'BYDDY'},
    'Sentiment': {'positive'},
    'Reason': {"BYD posted an impressive 165% registration gain in January 2026 compared to the prior year and is expanding aggressively into European markets. The article describes BYD as an 'EV juggernaut' and notes that Chinese co

In [93]:
system_prompt="""You are a helpfull assistant for "InvestAI" which helps people to give bunch of information about 
specific stocks,their price per share,total valume in market and gives news sourse for each stock.Only if user asks information about any ticker(stock)
please make sure that user parses name of the stock and the date.You are not able to share information about 
price per share for today's market,however you can share news that related today
.Additionally,when you share news please show tickers,give a sentiment analysis and its reason for various stocks which news are related to.
Please,if user asks something which you don't know,say so!

Super Important:
It is obvious that stock market closes at the saturdays and sundays.If user asks anything for the the weekends
 ,please tell user that it is the weekend date which market is not open!!
"""

In [94]:
def handle_tool_call(message):
    responses=[]

    for tool_call in message.tool_calls:
        func_name=tool_call.function.name
        args=json.loads(tool_call.function.arguments)
    
        if func_name=="about_stock":
         stock=args.get("stock_name")
         date=args.get("date")
         result=about_stock(stock.upper(),date)
        
        if func_name=="about_news":
            stock_name=args.get("stock_name")
            date_news=args.get("date")
            result=about_news(stock_name,date_news)

        content=json.dumps(result,ensure_ascii=False) if isinstance(result,dict) else str(result)

        responses.append({
            "role":"tool",
            "content":content,
            "tool_call_id":tool_call.id
         })

    return responses 


In [95]:
def chat_invest(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response=openai.chat.completions.create(
        model=openai_model,
        messages=messages,
        tools=tools
    )
    
    while response.choices[0].finish_reason=="tool_calls":
        message=response.choices[0].message
        response=handle_tool_call(message)
        messages.append(message)
        messages.extend(response)
        response=openai.chat.completions.create(model=openai_model, messages=messages)
       
    return response.choices[0].message.content


In [96]:
gr.ChatInterface(fn=chat_invest,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
